# - Parte 1

In [0]:
from pyspark.sql import SparkSession
tbl_produtos = spark.read.format("csv").option("header", "true").option("delimiter", ",").load("/Volumes/lh_nautical/datalake/datas/produtos_raw.csv")
tbl_produtos.display()

In [0]:
from pyspark.sql.functions import col, trim, lower, regexp_replace
from pyspark.sql.functions import when, col
#Tratando os dados
tbl_produtos =tbl_produtos.withColumn("actual_category", regexp_replace(lower(trim(col("actual_category"))), "[^a-zA-Z0-9]", ""))

#Padronizando os dados da coluna actual_category
tbl_produtos= tbl_produtos.withColumn("actual_category", when(tbl_produtos.actual_category.startswith("ele"), "Eletrônicos").when(tbl_produtos.actual_category.startswith("prop"), "Propulsão").when(tbl_produtos.actual_category.startswith("anc") | tbl_produtos.actual_category.startswith("en"), "Ancoragem"))

tbl_produtos.display()

# - Parte 2

In [0]:
from pyspark.sql.functions import when, col
import pyspark.sql.functions as F
from pyspark.sql.types import *
#Separando o preço e a moeda
split = F.split(tbl_produtos['price'], " ")
tbl_produtos = tbl_produtos.withColumn('monetary_unit', split.getItem(0)).withColumn('price', split.getItem(1))
#Modificando o tipo
tbl_produtos.withColumn('code', col("code").cast(IntegerType())).withColumn('price', col("price").cast(DecimalType(10,2))).display()

# - Parte 3

In [0]:
tbl_produtos.dropDuplicates(['name']).display()